# Week 3: LangGraph + RAG — Classification Pipeline

Last week we extracted entities from transcripts. This week we build a **multi-step pipeline** using LangGraph that:
1. Extracts entities from a transcript (what we did last week)
2. Uses RAG to match the problem to the correct **problem code** from our database

**What you'll learn:**
- LangGraph: nodes, edges, state, and how to wire them together
- RAG: embeddings, vector stores (ChromaDB), and similarity search
- How to combine extraction + retrieval in one workflow

## Setup

In [27]:
import json
import time
import warnings
from typing import Optional
from enum import Enum
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# LangChain: LLM wrapper and embedding model for converting text → vectors
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ChromaDB: open-source vector database — stores embeddings and supports similarity search
from langchain_chroma import Chroma

# Document: LangChain's wrapper for a chunk of text + metadata (used by the vector store)
from langchain_core.documents import Document

# LangGraph: graph-based workflow orchestration — nodes, edges, and shared state
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

# Suppress harmless Pydantic serialization warnings from LangChain's structured output
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

# Load OPENAI_API_KEY from .env file
load_dotenv()

True

## Part 1: Set up the RAG vector store

We'll load our 21 problem codes into ChromaDB so we can search them by similarity.

**Why RAG?** We could just list all 21 codes in the prompt, but imagine CBRE has 500 codes. RAG lets us find the relevant ones without stuffing everything into the context window.

In [28]:
# Load the 21 CBRE problem codes — these are the "knowledge base" our RAG system will search.
# Each code has a category, subcategory, description, severity, and keywords.
with open("problem_codes.json") as f:
    problem_codes = json.load(f)

print(f"Loaded {len(problem_codes)} problem codes")
print(f"\nExample: {problem_codes[0]['code']} — {problem_codes[0]['subcategory']}")
print(f"  {problem_codes[0]['description']}")

Loaded 21 problem codes

Example: PLUMB-001 — Pipe Leak / Burst Pipe
  Water leaking or flooding from pipes, including burst pipes, leaking joints, and water supply line failures. May involve ceiling leaks, wall leaks, or floor flooding.


In [29]:
# Step 1: Convert each problem code into a Document — one chunk per code.
# We combine the code, category, description, and keywords into a single text block
# so the embedding captures the full meaning of each problem code.

# the chunking process
documents = []
for pc in problem_codes:
    text = f"{pc['code']}: {pc['category']} — {pc['subcategory']}\n{pc['description']}\nKeywords: {', '.join(pc['keywords'])}"
    doc = Document(page_content=text, metadata={"code": pc["code"], "category": pc["category"]})
    documents.append(doc)


# Step 2: Create embeddings and index into ChromaDB.
# OpenAI's text-embedding-3-small converts each document into a vector.
# Chroma stores these vectors and lets us search by semantic similarity.
# NOTE: Without a persist_directory argument, Chroma runs entirely in-memory —
# no files are created on disk. The vectors only exist for this kernel session.
# Re-running this cell re-embeds all 21 docs (fast + cheap at this scale).
# For 500+ docs you'd add persist_directory="./chroma_db" to save to disk.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents, embeddings, collection_name="problem_codes")

# Step 3: Create a retriever that returns the top-k most similar codes for any query.
# k=3 means "give me the 3 closest matches." Try changing this to see how it affects results.
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector store ready with", len(documents), "problem codes")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************gIEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
# Quick test: does the retriever find the right codes for a water-related query?
# This is a sanity check — we expect plumbing codes to come back, not HVAC or elevator codes.
# Note: we're searching by MEANING, not keywords. "pipe burst" matches "burst pipe" even
# though the words are in different order. That's the power of embedding-based retrieval.
test_results = retriever.invoke("water leaking from ceiling pipe burst")
for doc in test_results:
    print(f"  [{doc.metadata['code']}] {doc.page_content[:80]}...")

  [PLUMB-001] PLUMB-001: Plumbing — Pipe Leak / Burst Pipe
Water leaking or flooding from pipe...
  [PLUMB-001] PLUMB-001: Plumbing — Pipe Leak / Burst Pipe
Water leaking or flooding from pipe...
  [PLUMB-001] PLUMB-001: Plumbing — Pipe Leak / Burst Pipe
Water leaking or flooding from pipe...
  [PLUMB-002] PLUMB-002: Plumbing — Roof Leak
Water intrusion from the roof due to damaged roo...
  [PLUMB-002] PLUMB-002: Plumbing — Roof Leak
Water intrusion from the roof due to damaged roo...


## Part 2: Define the LangGraph state

The **state** is a dictionary that flows through the graph. Each node reads from it and writes back to it. Think of it as the shared memory for the pipeline.

In [ ]:
# --- Pydantic models: tell the LLM exactly what shape of data to return ---

class Severity(str, Enum):
    CRITICAL = "Critical"
    HIGH = "High"
    MEDIUM = "Medium"
    LOW = "Low"


# MaintenanceEntity: the structured output from the extraction node.
# The LLM will fill in every field based on the transcript.
# This is the same schema we used in Week 2's entity extraction demo.
class MaintenanceEntity(BaseModel):
    problem_type: str = Field(description="Brief description of the maintenance problem")
    location_building: str = Field(description="Name of the building")
    location_detail: Optional[str] = Field(description="Specific location within the building")
    severity: Severity = Field(description="Critical / High / Medium / Low")
    caller_role: Optional[str] = Field(description="Role of the caller")
    urgency_indicators: list[str] = Field(description="Phrases indicating urgency")
    summary: str = Field(description="One-sentence summary")


# --- LangGraph state: the shared dictionary that flows through the entire graph ---
# Each node reads what it needs and writes back its results.
# Think of this as the pipeline's memory — every intermediate result lives here.
class PipelineState(TypedDict):
    transcript: str                             # input: raw call transcript
    entities: Optional[dict]                    # after extract_entities node
    retrieved_codes: Optional[list[dict]]       # after retrieve_codes node
    classification: Optional[dict]              # after classify_problem node

## Part 3: Build the nodes

Each node is just a Python function that takes state and returns updates to state.

In [ ]:
# temperature=0 for deterministic outputs — we want consistent extractions, not creative ones.
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

EXTRACTION_PROMPT = """
You are an expert building maintenance call analyst. Extract key information
from this transcript.

Severity guidelines:
- Critical: Immediate danger to life/safety (gas leak, fire, trapped persons, flooding)
- High: Significant disruption or escalation risk (major leak, broken security glass, HVAC failure)
- Medium: Needs attention soon, not emergency (broken door, minor plumbing, elevator malfunction)
- Low: Minor/cosmetic (flickering light, carpet stain, empty soap dispenser)

Transcript:
{transcript}
"""


def extract_entities(state: PipelineState) -> dict:
    """Node 1: Extract entities from the transcript.

    Reads: state["transcript"]
    Writes: state["entities"]

    Uses with_structured_output() to force the LLM to return data matching
    the MaintenanceEntity schema — no free-form text, just structured fields.
    """
    extractor = llm.with_structured_output(MaintenanceEntity)
    result = extractor.invoke(EXTRACTION_PROMPT.format(transcript=state["transcript"]))
    # .model_dump() converts the Pydantic object to a plain dict for the state
    return {"entities": result.model_dump()}

In [ ]:
def retrieve_codes(state: PipelineState) -> dict:
    """Node 2: Use RAG to find matching problem codes.

    Reads: state["entities"]  (from the extract step)
    Writes: state["retrieved_codes"]

    Builds a search query from the extracted problem type + summary,
    then retrieves the top-k most similar problem codes from ChromaDB.
    This is the RAG step — instead of showing the LLM all 21 codes,
    we only show the 3 most relevant ones.
    """
    entities = state["entities"]
    # Combine problem_type and summary into a single query for semantic search.
    # e.g. "water leak  Water is pouring from the ceiling near suite 310"
    query = f"{entities['problem_type']} {entities['summary']}"

    docs = retriever.invoke(query)
    codes = [{"code": d.metadata["code"], "content": d.page_content} for d in docs]
    return {"retrieved_codes": codes}

In [ ]:
CLASSIFICATION_PROMPT = """
You are classifying a building maintenance issue. Based on the extracted
information and the candidate problem codes retrieved from our database,
select the best matching code.

Extracted information:
- Problem: {problem_type}
- Severity: {severity}
- Summary: {summary}

Candidate problem codes:
{candidates}

Respond with your classification.
"""


# Classification: the structured output from the classify node.
# The LLM must pick a code, state its confidence, and explain why.
#
# IMPORTANT: The confidence score is SELF-REPORTED by the LLM — the model decides
# how confident it feels. This is NOT a calibrated probability or a similarity score.
# LLMs tend to be overconfident (you'll often see 0.90+ even on wrong predictions).
# It's useful as a relative signal (0.5 vs 0.95 means something), but don't treat
# it as a true probability. A more robust approach would use retriever similarity
# scores or logprobs — something to think about for production systems.
class Classification(BaseModel):
    selected_code: str = Field(description="The problem code that best matches (e.g., PLUMB-001)")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    reasoning: str = Field(description="Brief explanation of why this code was selected")


def classify_problem(state: PipelineState) -> dict:
    """Node 3: Classify the problem using extracted entities + retrieved codes.

    Reads: state["entities"] + state["retrieved_codes"]
    Writes: state["classification"]

    This is the "generation" step of RAG — the LLM sees the extracted info
    AND the top-k retrieved codes, then picks the best match. Without RAG,
    we'd have to stuff all 21 codes (or 500) into this prompt.
    """
    entities = state["entities"]
    codes = state["retrieved_codes"]

    # Format the retrieved codes as a readable block for the prompt
    candidates = "\n\n".join(c["content"] for c in codes)

    classifier = llm.with_structured_output(Classification)
    result = classifier.invoke(CLASSIFICATION_PROMPT.format(
        problem_type=entities["problem_type"],
        severity=entities["severity"],
        summary=entities["summary"],
        candidates=candidates,
    ))
    return {"classification": result.model_dump()}

## Part 4: Wire up the LangGraph

Now we connect the three nodes into a sequential pipeline:

```
START → extract_entities → retrieve_codes → classify_problem → END
```

In [ ]:
# Create the graph with PipelineState as the shared state type.
workflow = StateGraph(PipelineState)

# Register each function as a named node in the graph.
workflow.add_node("extract_entities", extract_entities)
workflow.add_node("retrieve_codes", retrieve_codes)
workflow.add_node("classify_problem", classify_problem)

# Wire the edges: START → extract → retrieve → classify → END
# This is a linear pipeline today. In Week 4 we'll add conditional edges
# (e.g., "if confidence < 0.7, route to human review instead of END").
workflow.add_edge(START, "extract_entities")
workflow.add_edge("extract_entities", "retrieve_codes")
workflow.add_edge("retrieve_codes", "classify_problem")
workflow.add_edge("classify_problem", END)

# compile() validates the graph and returns a runnable pipeline.
# After this, we can call pipeline.invoke({...}) to run the full flow.
pipeline = workflow.compile()

print("Pipeline compiled successfully!")

Pipeline compiled successfully!


## Part 5: Run the pipeline!

In [ ]:
# Load the 10 test transcripts — each has a ground truth category we can compare against.
with open("transcripts.json") as f:
    transcripts = json.load(f)

# Run the full pipeline on a single transcript to see each step's output.
# pipeline.invoke() sends the transcript through: extract → retrieve → classify
# and returns the complete state dict with all intermediate results.
test_transcript = transcripts[0]
print("INPUT:", test_transcript["transcript"][:100], "...")
print()

result = pipeline.invoke({"transcript": test_transcript["transcript"]})

# --- Show what each node produced ---

print("EXTRACTION:  (from extract_entities node)")
print(f"  Problem:  {result['entities']['problem_type']}")
print(f"  Severity: {result['entities']['severity']}")
print(f"  Summary:  {result['entities']['summary']}")
print()

print("RETRIEVED CODES:  (from retrieve_codes node — top 3 via RAG)")
for c in result["retrieved_codes"]:
    print(f"  [{c['code']}] {c['content'][:60]}...")
print()

print("CLASSIFICATION:  (from classify_problem node)")
print(f"  Code:       {result['classification']['selected_code']}")
print(f"  Confidence: {result['classification']['confidence']}")
print(f"  Reasoning:  {result['classification']['reasoning']}")
print()

# Compare against the ground truth label from our test set
print(f"  Actual:     {test_transcript['true_category']}")
print(f"  Match:      {'YES' if result['classification']['selected_code'] == test_transcript['true_category'] else 'NO'}")

INPUT: Hi, I'm calling from the third floor of the Westfield office building at 200 Main Street. There's wa ...



AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************gIEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

## Part 6: Evaluate across all transcripts

In [ ]:
def classify_one(t):
    """Run the full 3-node pipeline on a single transcript and compare to ground truth."""
    output = pipeline.invoke({"transcript": t["transcript"]})
    predicted = output["classification"]["selected_code"]
    actual = t["true_category"]
    match = predicted == actual
    return {
        "id": t["id"],
        "predicted": predicted,
        "actual": actual,
        "confidence": output["classification"]["confidence"],
        "match": match,
    }

# Each transcript triggers 2 LLM calls (extract + classify) plus 1 retrieval.
# Running them sequentially on 10 transcripts = 20+ LLM calls one after another.
# Since LLM calls are I/O-bound (waiting for the API), we parallelize across
# transcripts using ThreadPoolExecutor — all 10 run concurrently.
start = time.time()
all_results = []

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(classify_one, t): t for t in transcripts}
    for future in as_completed(futures):
        r = future.result()
        status = "CORRECT" if r["match"] else "WRONG"
        print(f"{r['id']}: predicted={r['predicted']:10s} actual={r['actual']:10s} conf={r['confidence']:.2f}  {status}")
        all_results.append(r)

elapsed = time.time() - start

# Results may print out of order (whichever transcript finishes first prints first)
# — that's expected with parallel execution.
correct = sum(1 for r in all_results if r["match"])
print(f"\nClassification accuracy: {correct}/{len(all_results)} ({100*correct/len(all_results):.0f}%)")

avg_conf = sum(r["confidence"] for r in all_results) / len(all_results)
print(f"Average confidence: {avg_conf:.2f}")
print(f"Wall-clock time: {elapsed:.1f}s (parallel across {len(transcripts)} transcripts)")

TX-004: predicted=HVAC-001   actual=HVAC-001   conf=0.95  CORRECT
TX-003: predicted=ELEV-001   actual=ELEV-001   conf=0.98  CORRECT
TX-002: predicted=ELEC-002   actual=ELEC-002   conf=0.95  CORRECT
TX-005: predicted=DOOR-002   actual=DOOR-002   conf=0.95  CORRECT
TX-001: predicted=PLUMB-001  actual=PLUMB-001  conf=0.95  CORRECT
TX-006: predicted=JANI-001   actual=JANI-001   conf=0.95  CORRECT
TX-007: predicted=SAFE-001   actual=SAFE-001   conf=0.98  CORRECT
TX-008: predicted=JANI-002   actual=JANI-002   conf=0.95  CORRECT
TX-009: predicted=PLUMB-002  actual=PLUMB-002  conf=0.95  CORRECT
TX-010: predicted=DOOR-001   actual=DOOR-001   conf=0.98  CORRECT

Classification accuracy: 10/10 (100%)
Average confidence: 0.96
Wall-clock time: 27.4s (parallel across 10 transcripts)


## Things to Explore (Optional)

Try these during or after the demo:

1. **Change `k` in the retriever** (`search_kwargs={"k": 5}`) — does retrieving more codes help or hurt accuracy?
2. **Look at the failures** — which transcripts get misclassified? Why? Is it a retrieval problem or a classification problem?
3. **Try without RAG** — put all 21 codes directly in the prompt. Compare accuracy and think about what happens at 500 codes.

## Assignment: Build Your Own Agentic RAG System

Follow the official LangChain tutorial and adapt it to the CBRE problem:

**Tutorial:** [Build a Custom RAG Agent with LangGraph](https://docs.langchain.com/oss/python/langgraph/agentic-rag)

The tutorial adds **document grading** and **query rewriting** — an agentic loop on top of basic RAG.
Your task: swap the blog posts for CBRE `problem_codes.json` and adapt the system for maintenance call classification.

See the `presentation.pdf` for full milestones and details.